# 04 — LLM Triple Scoring with Prompt Templates

Score candidate triples using all 5 Jinja2 templates and compare results.

**Requires API key** (`AI_GATEWAY_API_KEY` in `.env`).

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from src.config import Settings
from src.models.llm_client import LLMClient
from src.models.scorer import LLMScorer
from src.prompts.renderer import PromptRenderer
from src.wikidata.sparql import WikidataGrounder
from src.wikidata.cache import SPARQLCache
from src.utils.cost_tracker import CostTracker
from src.data.fb15k237 import FB15k237Dataset
from src.models.embedding_baseline import RotatEBaseline
from src.eval.candidates import generate_candidates

settings = Settings()
assert settings.ai_gateway_api_key, 'Set AI_GATEWAY_API_KEY in .env first!'
print(f'Model: {settings.ai_gateway_model}')

## 1. Initialise Components

In [ ]:
client   = LLMClient(settings=settings)
renderer = PromptRenderer()
cache    = SPARQLCache(cache_dir=settings.cache_dir)
grounder = WikidataGrounder(
    sparql_url=settings.wikidata_sparql_url,
    user_agent=settings.wikidata_user_agent,
    cache=cache,
)
tracker  = CostTracker()
scorer   = LLMScorer(client=client, renderer=renderer, cost_tracker=tracker)
print('Available templates:', renderer.list_templates())

## 2. Pick a Sample Query

In [ ]:
dataset = FB15k237Dataset(data_dir=settings.data_dir)
dataset.load()
h, r, t_true = dataset.test[0]
h_info = grounder.ground(h)
t_info = grounder.ground(t_true)
print(f'Head  : {h}  ({h_info.get("label","?")})')
print(f'Rel   : {r}')
print(f'Answer: {t_true} ({t_info.get("label","?")})')

## 3. Generate Candidates

In [ ]:
embed_model = RotatEBaseline(cache_dir=settings.cache_dir)
embed_model.load()
candidates = generate_candidates(
    model=embed_model, head=h, relation=r,
    dataset=dataset, top_k=settings.num_candidates,
)
print(f'Generated {len(candidates)} candidates.')

## 4. Score with All Five Templates

In [ ]:
rows = []
for tname in renderer.list_templates():
    scores = scorer.score_candidates(
        head=h, relation=r,
        candidates=[e for e,_ in candidates],
        grounder=grounder, template_name=tname,
    )
    ranked = sorted(scores.items(), key=lambda x:-x[1])
    true_rank = next((i+1 for i,(e,_) in enumerate(ranked) if e==t_true), None)
    rows.append({'template':tname,'true_rank':true_rank,'top1':ranked[0][0] if ranked else None})
    print(f'  {tname:<25} true_rank={true_rank}')
display(pd.DataFrame(rows))

## 5. Cost Breakdown by Template

In [ ]:
breakdown = tracker.breakdown()
print(f'Total calls: {tracker.total_calls}   Total cost: ${tracker.total_cost:.5f}')
if breakdown:
    fig,ax = plt.subplots(figsize=(8,4))
    ax.bar(breakdown.keys(), [v['cost'] for v in breakdown.values()], color='steelblue')
    ax.set(title='Cost per Template', ylabel='USD')
    plt.xticks(rotation=20,ha='right'); plt.tight_layout(); plt.show()

## 6. Inspect Rendered Prompts

In [ ]:
for tname in renderer.list_templates():
    prompt = renderer.render(
        template_name=tname,
        head=h_info.get('label',h), relation=r,
        candidate=t_info.get('label',t_true),
        head_description=h_info.get('description',''),
        candidate_description=t_info.get('description',''),
    )
    print(f'\n{"="*60}\nTEMPLATE: {tname}\n{"="*60}')
    print(prompt[:400], '...' if len(prompt)>400 else '')